# JIE2019 数据处理与校准

这份 notebook 主要内容包括：

1. 读取作者 `Data/Calibration` 中的原始 Excel。
2. 整理出模型需要的国家变量、贸易矩阵和投入产出变量。
3. 执行基准模型、静态合并模型、两国模型的参数校准。
4. 输出 CSV，而不是 MATLAB 的 `.mat` 文件。

## 0. 原始文件与输出文件

原始数据不能由程序生成，必须来自作者提供的 Excel。Notebook 会把中间结果和校准结果输出到 `自我复刻代码/CSV_数据处理与校准输出`。

In [23]:
from pathlib import Path

import numpy as np
import polars as pl
from scipy.optimize import least_squares
from scipy.special import gamma

DATA_DIR = Path(r"E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\原初数据")
OUTPUT_DIR = DATA_DIR.parent / "CSV_数据处理与校准输出"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

key = pl.read_excel(DATA_DIR / "Key.xlsx", has_header=False)
country_codes = key.get_column("column_1").head(44).drop_nulls().cast(pl.Utf8).to_list()
sector_names = key.get_column("column_2").head(4).drop_nulls().cast(pl.Utf8).to_list()

### 处理国家表

In [24]:
# ---------- PWT：人口、就业、GDP、资本和价格 ----------

pwt = (
    pl.read_excel(DATA_DIR / "pwt90.xlsx", sheet_name="Data", schema_overrides={"NX/GDP": pl.Float64})
    .filter(pl.col("year") == 2014)
)[:-1]
# 这里提供了总的，消费，投资，出口和进口的价格水平
pwt=pwt.with_columns(
    (pl.col('cgdpo')*pl.col('csh_i')).alias('investment')
)
# 为了计算ROW的价格和产出水平，我们需要进行一定的加总，比如将GDP和价格水平相乘，姑且成为中间变量
pwt=pwt.with_columns(
    (pl.col("cgdpo") * pl.col("pl_gdpo")).alias("py_weighted"),
    (pl.col("cgdpo") * pl.col("pl_c")).alias("pc_weighted"),
    (pl.col("cgdpo") * pl.col("pl_i")).alias("pi_weighted"),
    (pl.col("cgdpo") * pl.col("pl_x")).alias("px_weighted"),
    (pl.col("cgdpo") * pl.col("pl_m")).alias("pm_weighted")
)


Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 34, falling back to string
Could not determine dtype for column 47, falling back to string
Could not determine dtype for column 55, falling back to string


In [25]:
# key里面提供了我们关心的主要国家的变量，剩下的一律视为ROW，那么如何处理呢？
# 首先，我们最关心一个地区的人口，就业人口，GDP，资本和价格水平，所以我们需要对这些变量进行加总
pwt_sum=["pop", "emp", "cgdpo", "ck","investment", "py_weighted", "pc_weighted", "pi_weighted", "px_weighted", "pm_weighted"]
key_country=[code for code in country_codes if code !="ROW"]
selected_pwt=pwt.filter(pl.col("countrycode").is_in(key_country)).select(["countrycode",*pwt_sum])


In [26]:
# 接下来处理ROW的问题
world_sum=pwt.select([pl.sum(col).alias(col) for col in pwt_sum] ).row(0, named=True)
selected_sum=selected_pwt.select([pl.sum(col).alias(col) for col in pwt_sum] ).row(0, named=True)
ROW=pl.DataFrame([{"countrycode": "ROW"} | {col:world_sum[col]-selected_sum[col] for col in pwt_sum}])
#我们发现下面为什么到转过来的，原因在于ROW的计算逻辑
country_table=(
    pl.concat([selected_pwt,ROW])
    .rename({"countrycode":"country"})
    .with_columns(
        (pl.col('investment')/pl.col('cgdpo')).alias("csh_i"),
        (pl.col("py_weighted")/(pl.col("cgdpo"))).alias("pl_gdpo"),
        (pl.col("pc_weighted")/(pl.col("cgdpo"))).alias("pl_c"),
        (pl.col("pi_weighted")/(pl.col("cgdpo"))).alias("pl_i"),
        (pl.col("px_weighted")/(pl.col("cgdpo"))).alias("pl_x"),
        (pl.col("pm_weighted")/(pl.col("cgdpo"))).alias("pl_m"), 
    )
    .with_columns(((pl.col("pl_m")+pl.col("pl_x"))/2).alias("P_m"))
    .join(pl.DataFrame({"country": country_codes}), on="country", how="right")
    .select(["country", "pop", "emp", "cgdpo", "ck", "csh_i", "pl_gdpo", "pl_c", "pl_i", "pl_x", "pl_m", "P_m"])
)

In [27]:
# 部门生产要简单的多，无非就是转置一下生成一些中间表。最后合并进去
production = pl.read_excel(DATA_DIR / "Production.xlsx")
# 要把这个数据变为转置
wide_va=(
    production
    .group_by(["DestinationCode","DestinationCategory"])
    .agg(pl.sum('Value_VA').alias("VA"),
         pl.sum('Value_GO').alias("GO"))
    .pivot(index="DestinationCode", columns="DestinationCategory", values=["VA", "GO"], aggregate_function="sum")
    .rename({"DestinationCode": "country"})
    .with_columns(
        (pl.col("VA_Durable goods") + pl.col("VA_Durable services") + pl.col("VA_Nondurable goods") + pl.col("VA_Nondurable services")).alias("GDP"),
        (pl.col("VA_Durable goods") + pl.col("VA_Durable services")).alias("VA_x"),
        pl.col("VA_Nondurable goods").alias("VA_m"),
        pl.col("VA_Nondurable services").alias("VA_c"),
        (pl.col("GO_Durable goods") + pl.col("GO_Durable services")).alias("GO_x"),
        pl.col("GO_Nondurable goods").alias("GO_m"),
        pl.col("GO_Nondurable services").alias("GO_c"),
    )
    .select(["country", "GDP", "VA_x", "VA_m", "VA_c", "GO_x", "GO_m", "GO_c"])
)


C:\Users\27968\AppData\Local\Temp\ipykernel_28932\3953511422.py:9: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index="DestinationCode", columns="DestinationCategory", values=["VA", "GO"], aggregate_function="sum")


In [28]:
country_table=country_table.join(wide_va,on="country",how='left')

In [29]:
final_expenditures = pl.read_excel(DATA_DIR / "FinalExpenditures.xlsx")
final_expenditures=(
    final_expenditures
    .with_columns(
        pl.when(pl.col("DestinationCategory")=="Investment")
        .then(pl.lit("GFCF"))
        .otherwise(pl.lit("Con"))
        .alias('demand_type')
    )
    .group_by(["DestinationCountryCode", "demand_type"])
    .agg(pl.sum('Industry_Sum').alias("value"))
    .pivot(index="DestinationCountryCode",columns="demand_type",values="value",aggregate_function="sum")
    .rename({"DestinationCountryCode":"country"})
)

C:\Users\27968\AppData\Local\Temp\ipykernel_28932\792796417.py:12: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index="DestinationCountryCode",columns="demand_type",values="value",aggregate_function="sum")


In [30]:
country_table=country_table.join(
    final_expenditures,on="country",how='left'
)

In [31]:
# 接着处理IO表
io_matrix = pl.read_excel(DATA_DIR / "IOmatrix.xlsx")
# 要划分九个IO块，IOab表示a生成中来自b的部分
io_table=(
    io_matrix
    .with_columns(
        pl.when(pl.col('OriginCategory')=="Nondurable goods")
        .then(pl.lit('m'))
        .when(pl.col('OriginCategory')=="Nondurable services")
        .then(pl.lit("c"))
        .otherwise(pl.lit("x"))
        .alias("origin_sector"),

        pl.when(pl.col("DestinationCategory") == "Nondurable goods")
        .then(pl.lit("m"))
        .when(pl.col("DestinationCategory") == "Nondurable services")
        .then(pl.lit("c"))
        .otherwise(pl.lit("x"))
        .alias("dest_sector"),
    )
    .group_by(["OriginCountryCode","origin_sector","dest_sector"])
    .agg(pl.sum("Value").alias("value"))
    .with_columns(
        (
            pl.lit("IO_")+pl.col("origin_sector")+pl.col("dest_sector")
        )
        .alias("io_name")
    )
    .pivot(index="OriginCountryCode",columns="io_name",values="value",aggregate_function="sum")
    .rename({"OriginCountryCode":"country"})
)

C:\Users\27968\AppData\Local\Temp\ipykernel_28932\1813590617.py:29: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index="OriginCountryCode",columns="io_name",values="value",aggregate_function="sum")


In [32]:
country_table=country_table.join(
    io_table,on="country",how='left'
)

### 处理贸易表

In [33]:
trade=(
    pl.read_excel(DATA_DIR / "Trade.xlsx")
    .filter(pl.col("Category")=="Nondurable goods")
    .rename({"ScaledValue":"trade","DestinationCode":"destination","OriginCode":"origin"})
    .drop(pl.col("Category"))
    .with_columns(pl.col("destination").alias("country"))
)
exports=(
    trade
    .filter(pl.col("destination")!=pl.col("origin"))
    .group_by("origin")
    .agg(pl.sum("trade").alias("export"))
   .rename({"origin":"country"})
)
imports=(
    trade
    .filter(pl.col("origin")!=pl.col("destination"))
    .group_by("destination")
    .agg(pl.sum("trade").alias("import"))
    .rename({"destination":"country"})
)
country_table=(
    country_table
    .join(exports,on="country",how="left")
    .join(imports,on="country",how="left")
    .with_columns((pl.col("GO_m")-pl.col("export")+pl.col("import")).alias("ABS"))
)
trade_table=(
    trade
    .join(country_table.select(["country","ABS"]),on="country",how="left")
    .with_columns((pl.col("trade")/pl.col("ABS")).alias("trade_share"))
    .drop("ABS")
)

#### 归一化
上述代码计算出来经济变量了，对其进行归一化处理

In [34]:
normal=country_table.select(pl.sum("GDP")).item()
value_columns = [
    "ABS", "export", "GDP", "GO_c", "GO_m", "GO_x", "import",
    "VA_c", "VA_m", "VA_x", "GFCF", "pl_c", "pl_i", "P_m",
    "IO_mm", "IO_mc", "IO_mx", "IO_cm", "IO_cc", "IO_cx", "IO_xm", "IO_xc", "IO_xx",
]
country_table=(
    country_table
    .with_columns([pl.col(col)/normal for col in value_columns])
)
trade_table = trade_table.with_columns((pl.col("trade") / normal).alias("trade"))

#### 推算贸易成本

In [35]:
theta = 4.0

domestic_share = (
    trade_table
    .filter(pl.col("destination") == pl.col("origin"))
    .select([
        pl.col("origin"),
        pl.col("trade_share").alias("domestic_share")
    ])
)

prices = country_table.select([
    pl.col("country"),
    pl.col("P_m").alias("price")
])

trade_table = (
    trade_table
    # 加上目的国价格 P_i
    .join(
        prices.rename({"country": "destination", "price": "dest_price"}),
        on="destination",
        how="left"
    )
    # 加上来源国价格 P_j
    .join(
        prices.rename({"country": "origin", "price": "origin_price"}),
        on="origin",
        how="left"
    )
    # 加上来源国自己的国内份额 π_jj
    .join(domestic_share, on="origin", how="left")
    .with_columns(
        pl.when(pl.col("destination") == pl.col("origin"))
        .then(1.0)
        .when(pl.col("trade_share") == 0)
        .then(10_000_000.0)
        .otherwise(
            (
                (pl.col("trade_share") / pl.col("domestic_share")) ** (-1.0 / theta)
                * (pl.col("dest_price") / pl.col("origin_price"))
            ).clip(lower_bound=1.0)
        )
        .alias("trade_cost")
    )
    .drop(["dest_price", "origin_price", "domestic_share"])
)

### 制作参数表

从这一步往后，就进入到了数值的领域，再使用polars就不太合适了，应当转为numpy

In [36]:
countries = country_table["country"].to_list()
n = len(countries)
base_pos = countries.index("USA")

df = (
    country_table
    .to_pandas()
    .set_index("country")
    .loc[countries]
)


#### 直接设定

In [37]:
theta = 4.0
eta = 2.0
alpha = np.full(n, 0.33)

beta = 0.96
sigma = 0.5
delta = 0.06
lambda_adjustment = 0.76
chi = delta ** (1 - lambda_adjustment)

gam = gamma(1 + (1 / theta) * (1 - eta)) ** (1 / (1 - eta))
Alpha = alpha ** (-alpha) * (1 - alpha) ** (alpha - 1)

#### 从数据中读取

In [38]:
L = df["emp"].to_numpy()
K = df["ck"].to_numpy()

nu_m = (df["VA_m"] / df["GO_m"]).to_numpy()
nu_x = (df["VA_x"] / df["GO_x"]).to_numpy()
nu_c = (df["VA_c"] / df["GO_c"]).to_numpy()

pl_i = df["pl_i"].to_numpy()
pl_c = df["pl_c"].to_numpy()
P_m = df["P_m"].to_numpy()
GDP = df["GDP"].to_numpy()

imports = df["import"].to_numpy()
exports = df["export"].to_numpy()

share=domestic_share.to_numpy()[:,1]

#### 校准

In [39]:
def cost_constant(alpha, value_added_share):
    return (
        (alpha * value_added_share) ** (-alpha * value_added_share)
        * ((1 - alpha) * value_added_share) ** ((alpha - 1) * value_added_share)
        * (1 - value_added_share) ** (value_added_share - 1)
    )
B_m = cost_constant(alpha, nu_m)
B_x = cost_constant(alpha, nu_x)
B_c = cost_constant(alpha, nu_c)

In [40]:
Z_base = (
    (1 / share[base_pos]) ** (1 / theta)
    / (gam * B_m[base_pos])
)

denom = np.array([
    B_x[base_pos]
    * Z_base ** (nu_x[base_pos] / nu_m[base_pos]),

    B_c[base_pos]
    * Z_base ** (nu_c[base_pos] / nu_m[base_pos]),

    Alpha[base_pos]
    * (1 / B_c[base_pos])
    * Z_base ** ((1 - nu_c[base_pos]) / nu_m[base_pos])
    * (K[base_pos] / L[base_pos]) ** alpha[base_pos],
])

base_target = np.array([
    pl_i[base_pos] / P_m[base_pos],
    pl_c[base_pos] / P_m[base_pos],
    GDP[base_pos] / L[base_pos] / pl_c[base_pos],
])

In [41]:
def calibrate_one_country(pos):
    #  数据直接读
    target= np.array([
    pl_i[pos] / P_m[pos],
    pl_c[pos] / P_m[pos],
    GDP[pos] / L[pos] / pl_c[pos],
    ]) / base_target

    # 模型直接算
    def residual(log_params):
        T_m, A_c, A_x = np.exp(log_params)
        Z=(T_m/share[pos])**(1/theta)/(gam*B_m[pos])
        model=np.array([
            B_x[pos]/A_x*Z**(nu_x[pos]/nu_m[pos]),
            B_c[pos]/A_c* Z ** (nu_c[pos] / nu_m[pos]),
            Alpha[pos]* (1 / B_c[pos])*  A_c*Z ** ((1 - nu_c[pos]) / nu_m[pos])* (K[pos] / L[pos]) ** alpha[pos],
        ])
        return model / denom - target
    fit = least_squares(
        residual,
        x0=np.zeros(3),
        ftol=1e-10,
        xtol=1e-10,
        gtol=1e-10,
        max_nfev=1_000_000,
    )

    if not fit.success:
        raise RuntimeError(f"{countries[pos]} 校准失败：{fit.message}")

    return fit.x



In [42]:
solutions = np.vstack([
    calibrate_one_country(pos)
    for pos in range(n)
])

T_m0 = np.exp(solutions[:, 0])
A_c0 = np.exp(solutions[:, 1])
A_x0 = np.exp(solutions[:, 2])

In [43]:
parameter_table = pl.DataFrame({
    "country": countries,
    "T_m0": T_m0,
    "A_c0": A_c0,
    "A_x0": A_x0,
    "alpha0": alpha,
    "nu_m0": nu_m,
    "nu_x0": nu_x,
    "nu_c0": nu_c,
    "L0": L,
    "K0": K,
    "F": imports - exports,
    "delta": np.full(n, delta),
    "lambda": np.full(n, lambda_adjustment),
    "chi": np.full(n, chi),
    "theta": np.full(n, theta),
    "eta": np.full(n, eta),
    "gam": np.full(n, gam),
    "beta": np.full(n, beta),
    "sigma": np.full(n, sigma),
})

parameter_table

country,T_m0,A_c0,A_x0,alpha0,nu_m0,nu_x0,nu_c0,L0,K0,F,delta,lambda,chi,theta,eta,gam,beta,sigma
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AUS""",0.899974,0.742233,0.686229,0.33,0.403086,0.31432,0.57146,11.96684,3944975.5,-0.000889,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""AUT""",0.464541,0.796171,0.931208,0.33,0.304954,0.376289,0.571025,4.420995,1.8910e6,0.000002,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""BEL""",0.390895,0.848303,1.049283,0.33,0.213468,0.290667,0.535369,4.899381,2.300616e6,-0.000539,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""BGR""",0.144643,0.931471,1.196715,0.33,0.26175,0.230074,0.517691,3.57134,312399.375,0.000072,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""BRA""",0.214221,0.466372,0.974406,0.33,0.309813,0.357782,0.637005,105.908615,1.3655093e7,-0.000628,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SVN""",0.153521,0.753373,1.056875,0.33,0.348866,0.312386,0.544971,0.96693,338292.25,0.000007,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""SWE""",0.714937,0.774125,0.796358,0.33,0.324531,0.411493,0.563208,4.751581,1.774148e6,-0.00012,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5
"""TUR""",0.220757,0.83346,1.313834,0.33,0.330949,0.368365,0.58988,24.647476,4605440.5,-0.000235,0.06,0.76,0.509045,4.0,2.0,0.816049,0.96,0.5


In [44]:
country_table.write_csv(OUTPUT_DIR / "01_baseline_country_table.csv")
trade_table.write_csv(OUTPUT_DIR / "02_baseline_trade_table.csv")
parameter_table.write_csv(OUTPUT_DIR / "03_baseline_parameter_table.csv")

pl.DataFrame({
    "file_name": [
        "01_baseline_country_table.csv",
        "02_baseline_trade_table.csv",
        "03_baseline_parameter_table.csv",
    ],
    "path": [
        str(OUTPUT_DIR / "01_baseline_country_table.csv"),
        str(OUTPUT_DIR / "02_baseline_trade_table.csv"),
        str(OUTPUT_DIR / "03_baseline_parameter_table.csv"),
    ],
})

file_name,path
str,str
"""01_baseline_country_table.csv""","""E:\zotero\部分论文复刻\JIE2019资本积累和动…"
"""02_baseline_trade_table.csv""","""E:\zotero\部分论文复刻\JIE2019资本积累和动…"
"""03_baseline_parameter_table.cs…","""E:\zotero\部分论文复刻\JIE2019资本积累和动…"
